[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TianshuangQiu/TorchCode/blob/master/solutions/42_topk_gather_solution.ipynb)

# 🟡 Solution: Top-k Gather

**Primitive: `topk` + `gather` with index expansion**

**Reduction:** `output[b, i]` = `values[b, j]` where `j` is the index of the `i`-th largest score in row `b`.

The key shape manipulation:
1. `torch.topk(scores, k, dim=-1).indices` → `(B, k)` — which columns to take
2. `unsqueeze(-1).expand(-1, -1, D)` → `(B, k, D)` — broadcast the column index across the feature dimension
3. `values.gather(dim=1, index)` → `(B, k, D)` — select

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def topk_gather(scores: torch.Tensor, values: torch.Tensor, k: int) -> torch.Tensor:
    # primitive: topk returns (B, k) indices; expand to (B, k, D) for gather
    _, idx = torch.topk(scores, k, dim=-1)                    # (B, k)
    idx = idx.unsqueeze(-1).expand(-1, -1, values.shape[-1])  # (B, k, D)
    return values.gather(dim=1, index=idx)                    # (B, k, D)

In [ ]:
# Verify
scores = torch.tensor([[0.1, 0.9, 0.4, 0.7]])
values = torch.tensor([[[1.,1.],[2.,2.],[3.,3.],[4.,4.]]])
result = topk_gather(scores, values, k=2)
print('output:', result.tolist())
print('expect: [[[2.0, 2.0], [4.0, 4.0]]]')
print('shape: ', result.shape)

In [ ]:
import torch, time

# ── Test 1: basic correctness (1, 4, 2) k=2 ───────────────────────────────
scores = torch.tensor([[0.1, 0.9, 0.4, 0.7]])
values = torch.tensor([[[1,1],[2,2],[3,3],[4,4]]]).float()
result = topk_gather(scores, values, k=2)
assert result.shape == (1, 2, 2), f"Shape: {result.shape}"
assert torch.allclose(result[0, 0], torch.tensor([2., 2.])), f"First: {result[0,0]}"
assert torch.allclose(result[0, 1], torch.tensor([4., 4.])), f"Second: {result[0,1]}"
print("Test 1 passed: basic correctness")

# ── Test 2: k=1 preserves all dimensions ──────────────────────────────────
torch.manual_seed(42)
B, N, D = 4, 10, 8
scores = torch.randn(B, N)
values = torch.randn(B, N, D)
result = topk_gather(scores, values, k=1)
assert result.shape == (B, 1, D), f"Shape with k=1: {result.shape}, expected ({B}, 1, {D})"
print("Test 2 passed: k=1 preserves dimensions")

# ── Test 3: k=N returns values ordered by score ───────────────────────────
torch.manual_seed(7)
B, N, D = 2, 5, 4
scores = torch.randn(B, N)
values = torch.randn(B, N, D)
result = topk_gather(scores, values, k=N)
assert result.shape == (B, N, D), f"Shape with k=N: {result.shape}"
for b in range(B):
    order = scores[b].argsort(descending=True)
    expected_row = values[b][order]
    assert torch.allclose(result[b], expected_row), f"Order wrong for batch {b}"
print("Test 3 passed: k=N returns all values in score order")

# ── Test 4: batch of 2, shape (2, 5, 3) k=2 ──────────────────────────────
torch.manual_seed(99)
scores = torch.randn(2, 5)
values = torch.randn(2, 5, 3)
result = topk_gather(scores, values, k=2)
assert result.shape == (2, 2, 3), f"Shape: {result.shape}"
topk_idx = scores.topk(2, dim=-1).indices
for b in range(2):
    for ki in range(2):
        expected = values[b, topk_idx[b, ki]]
        assert torch.allclose(result[b, ki], expected), f"Mismatch b={b} ki={ki}"
print("Test 4 passed: batch index correctness")

# ── Test 5: large input (32, 1000, 64) k=10 (timing) ─────────────────────
torch.manual_seed(0)
t0 = time.time()
scores = torch.randn(32, 1000)
values = torch.randn(32, 1000, 64)
result = topk_gather(scores, values, k=10)
elapsed = time.time() - t0
assert result.shape == (32, 10, 64), f"Shape: {result.shape}"
assert elapsed < 2.0, f"Too slow: {elapsed:.2f}s (expected <2s)"
print(f"Test 5 passed: large input timing ({elapsed:.3f}s)")

print("\nAll tests passed!")
